# Log a pipeline outcome

Append one parameterized success or failure record to `audit.pipeline_run_log`.

In [ ]:
run_id = "interactive"
pipeline_name = "etl"
activity_name = "manual"
status = "Succeeded"
error_code = ""
error_message = ""

## Append the run record

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

if status not in {"Succeeded", "Failed"}:
    raise ValueError("status must be Succeeded or Failed")

spark.sql("CREATE SCHEMA IF NOT EXISTS audit")
schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("pipeline_name", StringType(), False),
    StructField("activity_name", StringType(), False),
    StructField("status", StringType(), False),
    StructField("error_code", StringType(), False),
    StructField("error_message", StringType(), False),
])

record = [(
    str(run_id),
    str(pipeline_name),
    str(activity_name),
    str(status),
    str(error_code or "")[:500],
    str(error_message or "")[:4000],
)]

log_df = spark.createDataFrame(record, schema).withColumn("logged_at", F.current_timestamp())
log_df.write.format("delta").mode("append").saveAsTable("audit.pipeline_run_log")
print(f"Logged {status} for {pipeline_name}/{activity_name}, run {run_id}")